# SICK Vision Sensor Recommender System
### Comparing Collaborative Filtering, Content-Based Filtering, Knowledge-Based, and Hybrid Approaches

**Domain:** Field application engineers at SICK configure machine vision inspection systems for
semiconductor and electronics manufacturing lines (e.g. wafer defect detection, PCB solder
inspection, IC package marking verification). Instead of manually cross-referencing spec sheets
across SICK's full sensor portfolio, this notebook builds a recommender system that suggests the
right sensor for a given inspection task, based on how similar past engineers/projects were solved.

**Users** = field application engineers, each with a history of past project outcomes.
**Items** = SICK vision sensor models across real product families (Inspector, Lector, IVC-2D,
IVC-3D, Ranger, Ruler, TriSpector, 2D Vision Sensors).
**Interactions** = 1-5 ratings reflecting how well a given sensor performed for a given engineer's
past inspection tasks.

This notebook implements and compares **User-Based Collaborative Filtering** and
**Content-Based Filtering** on the same synthetic-but-realistic dataset, using only NumPy and
Matplotlib (no external recommender libraries), per assignment requirements.


## 1. Item Catalog

**Design decision:** rather than inventing 80 unrelated product names, we expand each **real**
SICK product family into multiple realistic variants -- this mirrors how SICK's actual catalog is
structured (one family, many SKUs varying by resolution/speed/housing). Each family's spec range
below is grounded in SICK's published vision-portfolio material (e.g. Inspector83x: up to 5MP with
built-in lighting and AI capability; Lector650: 2-4MP at up to 40Hz; Ranger/Ruler/TriSpector: 3D
profiling lines used for shape, volume, and surface inspection).

This is the "synthetic but justified" dataset the assignment calls for: **real product families and
real spec ranges**, systematically expanded to reach catalog scale. The one fully synthetic column
is `price_tier` (SICK doesn't publish per-unit pricing), included as a relative 1-5 estimate.

Six features are encoded per item -- chosen because each one is something that actually determines
whether a sensor **can** do a given inspection job, not a cosmetic attribute:
resolution (can it see the defect?), frame rate (can it keep up with the line?), 2D vs. 3D
(is depth information needed?), built-in lighting (self-contained or not?), AI-capability
(rule-based vs. learned inspection), and price tier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng_catalog = np.random.default_rng(42)  # fixed seed -> reproducible

# (family, dimension, resolution_range_mp, frame_rate_range_hz,
#  built_in_light, ai_capable, price_tier_range, n_variants)
families = [
    ("IVC-2D",          "2D", (0.3, 2.0),  (20, 30), 1, 0, (1, 2), 12),
    ("Inspector I83x",  "2D", (1.0, 5.0),  (10, 20), 1, 1, (3, 5), 10),
    ("Lector650",       "2D", (1.0, 4.0),  (30, 45), 0, 0, (2, 4), 8),
    ("2D Vision Sensor","2D", (1.0, 12.0), (8, 20),  0, 0, (2, 4), 12),
    ("2D Sensor Entry", "2D", (0.3, 1.0),  (15, 25), 1, 0, (1, 2), 8),
    ("IVC-3D",          "3D", (0.5, 2.0),  (15, 25), 0, 0, (3, 4), 8),
    ("Ranger3",         "3D", (1.0, 3.0),  (30, 60), 0, 0, (4, 5), 8),
    ("Ruler3000",       "3D", (0.5, 1.5),  (25, 40), 0, 0, (4, 5), 6),
    ("TriSpector",      "3D", (1.0, 2.0),  (15, 35), 0, 1, (4, 5), 8),
]

catalog = []
for fam, dim, res_range, fr_range, light, ai, price_range, n in families:
    for i in range(n):
        res = round(float(rng_catalog.uniform(*res_range)), 1)
        fr = int(rng_catalog.integers(fr_range[0], fr_range[1] + 1))
        price = int(rng_catalog.integers(price_range[0], price_range[1] + 1))
        name = f"{fam}-{res}MP-{fr}Hz"
        catalog.append((name, res, fr, dim, light, ai, price))

# de-duplicate names (rare collision from rounding)
seen = {}
final_catalog = []
for row in catalog:
    name = row[0]
    if name in seen:
        seen[name] += 1
        name = f"{name}-v{seen[name]}"
    else:
        seen[name] = 1
    final_catalog.append((name,) + row[1:])
catalog = final_catalog
names = [c[0] for c in catalog]
n_items = len(catalog)

print(f"Catalog size: {n_items} items")
n_2d = sum(1 for c in catalog if c[3] == "2D")
n_3d = sum(1 for c in catalog if c[3] == "3D")
print(f"2D items: {n_2d}, 3D items: {n_3d}")
print("\nSample rows (name, resolution_mp, frame_rate_hz, dim, built_in_light, ai_capable, price_tier):")
for row in catalog[:5]:
    print(" ", row)


## 2. Users and the Interaction Matrix

**Design decision:** each engineer is given a hidden **task profile** -- a *target* (ideal value)
and a *weight* (how much they care) per feature. A rating is generated from how close an item's
features are to the engineer's ideal, weighted by what they personally care about, plus noise.

This models something real: an engineer who's spent years on high-speed packaging inspection
genuinely rates high-frame-rate sensors higher, because that reflects what actually worked on
their past projects -- it is not arbitrary noise. Using a **target + weight** pair (rather than a
single preference-weight vector) matters mechanically too: with only positive-valued features,
plain similarity metrics can't express "this engineer wants LOW frame rate specifically," since two
vectors that are both positive can never point in meaningfully different directions. Measuring
distance from an explicit ideal fixes this and produces realistic variation in ratings (1 to 5,
not clustered at the top).

Each engineer only rates a random subset of 12-20 items (not all 80) -- real engineers have only
worked on a handful of past projects, not tried every SKU. Unrated cells are left at 0, which is
also realistic and is exactly what a recommender is trying to fill in.

In [ ]:
rng_users = np.random.default_rng(7)

engineers = [
    "E1_HighSpeedPackaging", "E2_WaferDefectPrecision", "E3_BudgetLineRetrofit",
    "E4_3DVolumeInspection", "E5_PCBSolderAI", "E6_GeneralAssemblyQA",
    "E7_HighResFineDefect", "E8_FastConveyor3D", "E9_LowCostEntryLine",
    "E10_AIRuleHybrid",
]

# profiles[engineer] = (target, weight), both over:
# [resolution, frame_rate, is_3D, built_in_light, ai_capable, price_tier]
profiles = {
    "E1_HighSpeedPackaging":   ([0.3, 0.9, 0.1, 0.5, 0.2, 0.4], [0.1, 0.9, 0.1, 0.2, 0.1, 0.2]),
    "E2_WaferDefectPrecision": ([0.95,0.2, 0.2, 0.4, 0.4, 0.8], [0.9, 0.2, 0.1, 0.2, 0.3, 0.3]),
    "E3_BudgetLineRetrofit":   ([0.3, 0.4, 0.0, 0.8, 0.0, 0.1], [0.2, 0.2, 0.1, 0.4, 0.1, 0.9]),
    "E4_3DVolumeInspection":   ([0.4, 0.5, 1.0, 0.0, 0.1, 0.6], [0.2, 0.3, 0.9, 0.1, 0.1, 0.3]),
    "E5_PCBSolderAI":          ([0.6, 0.4, 0.1, 0.6, 1.0, 0.5], [0.4, 0.2, 0.1, 0.3, 0.9, 0.3]),
    "E6_GeneralAssemblyQA":    ([0.5, 0.5, 0.3, 0.5, 0.3, 0.5], [0.3, 0.3, 0.3, 0.3, 0.3, 0.3]),
    "E7_HighResFineDefect":    ([1.0, 0.2, 0.2, 0.4, 0.3, 0.8], [0.95,0.1, 0.1, 0.2, 0.2, 0.3]),
    "E8_FastConveyor3D":       ([0.3, 0.8, 0.9, 0.0, 0.1, 0.6], [0.1, 0.7, 0.8, 0.1, 0.1, 0.3]),
    "E9_LowCostEntryLine":     ([0.2, 0.3, 0.0, 0.9, 0.0, 0.05],[0.2, 0.2, 0.1, 0.5, 0.1, 0.95]),
    "E10_AIRuleHybrid":        ([0.5, 0.4, 0.3, 0.4, 1.0, 0.4], [0.3, 0.2, 0.2, 0.2, 0.95,0.3]),
}

def item_feature_vector(row):
    """Normalize each raw catalog field to roughly [0, 1] so features are comparable."""
    _, res, fr, dim, light, ai, price = row
    return np.array([
        res / 12.0,                    # resolution, normalized against the highest in-catalog value
        fr / 60.0,                     # frame rate, normalized against the highest in-catalog value
        1.0 if dim == "3D" else 0.0,   # dimensionality, one-hot rather than an arbitrary 0/1 order
        float(light),
        float(ai),
        price / 5.0,
    ])

item_features = np.array([item_feature_vector(row) for row in catalog])  # shape (80, 6)

n_users = len(engineers)
ratings = np.zeros((n_users, n_items))
MIN_RATED, MAX_RATED = 12, 20  # each engineer has worked on 12-20 past projects

for u_idx, eng in enumerate(engineers):
    target, weight = np.array(profiles[eng][0]), np.array(profiles[eng][1])
    n_rated = rng_users.integers(MIN_RATED, MAX_RATED + 1)
    rated_items = rng_users.choice(n_items, size=n_rated, replace=False)
    for i_idx in rated_items:
        feat = item_features[i_idx]
        # weighted RMS distance between item features and engineer's ideal
        dist = np.sqrt(np.sum(weight * (target - feat) ** 2) / np.sum(weight))
        match_score = 1.0 - dist  # 1.0 = perfect fit, 0.0 = total mismatch
        rating = 1.0 + 4.0 * match_score + rng_users.normal(0, 0.35)
        ratings[u_idx, i_idx] = np.clip(round(rating), 1, 5)

print(f"Ratings matrix shape: {ratings.shape} (users x items)")
print(f"Sparsity: {100 * (ratings == 0).sum() / ratings.size:.1f}% unrated")


**Sanity check:** before trusting the recommenders, we confirm the synthetic ratings actually
have structure -- a healthy spread across 1-5 (not clustered at the ceiling), and different mean
ratings per engineer (reflecting their different task profiles).

In [ ]:
vals = ratings[ratings > 0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(vals, bins=[0.5,1.5,2.5,3.5,4.5,5.5], rwidth=0.8, color="#4c72b0")
axes[0].set_xticks([1,2,3,4,5])
axes[0].set_title("Overall rating distribution")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Count")

means = [ratings[u][ratings[u] > 0].mean() for u in range(n_users)]
axes[1].barh(engineers[::-1], means[::-1], color="#dd8452")
axes[1].set_title("Mean rating per engineer")
axes[1].set_xlabel("Mean rating")

plt.tight_layout()
plt.show()


## 3. User-Based Collaborative Filtering

**Approach:** to recommend for a target engineer, find the engineers whose *rating patterns* are
most similar (not their job titles -- their actual behavior), then recommend items those neighbors
rated highly that the target hasn't tried yet, weighted by neighbor similarity.

**Why cosine similarity:** it measures the angle between two rating vectors, ignoring magnitude --
so an engineer who rates everything a bit more generously overall is still matched correctly with
someone who shares the same *pattern* of likes and dislikes, even if their absolute numbers
differ.

In [ ]:
def cosine_sim_matrix(matrix):
    """Row-wise cosine similarity. 0 = unrated is treated as neutral/absent,
    which is the standard convention for sparse rating matrices."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1e-9
    normalized = matrix / norms
    return normalized @ normalized.T

user_sim = cosine_sim_matrix(ratings)

def top_k_similar_users(user_idx, k=3):
    sims = user_sim[user_idx].copy()
    sims[user_idx] = -1  # exclude self
    top_idx = np.argsort(sims)[::-1][:k]
    return [(engineers[i], sims[i]) for i in top_idx]

def cf_recommend(user_idx, k_neighbors=3, top_n=5):
    neighbors = top_k_similar_users(user_idx, k_neighbors)
    neighbor_idx = [engineers.index(n[0]) for n in neighbors]
    neighbor_sims = np.array([n[1] for n in neighbors])

    scores = np.zeros(n_items)
    sim_totals = np.zeros(n_items)
    for n_idx, sim in zip(neighbor_idx, neighbor_sims):
        for i in range(n_items):
            if ratings[n_idx, i] > 0:
                scores[i] += sim * ratings[n_idx, i]
                sim_totals[i] += abs(sim)

    already_rated = ratings[user_idx] > 0
    predicted = np.where(sim_totals > 0, scores / np.where(sim_totals == 0, 1, sim_totals), 0)
    predicted[already_rated] = -1  # never recommend what they've already used

    top_idx = np.argsort(predicted)[::-1][:top_n]
    return [(names[i], round(predicted[i], 2)) for i in top_idx], neighbors

target = "E1_HighSpeedPackaging"
target_idx = engineers.index(target)
cf_recs, cf_neighbors = cf_recommend(target_idx)

print(f"Target engineer: {target}\n")
print("Top-3 similar engineers (name, cosine similarity):")
for n, s in cf_neighbors:
    print(f"  {n}: {s:.3f}")
print(f"\nTop-5 CF recommendations for {target}:")
for name, score in cf_recs:
    print(f"  {name}: predicted score {score}")


## 4. Content-Based Filtering

**Approach:** build the target engineer's implied preference **profile** from the features of
items they rated, then rank every unrated item by cosine similarity to that profile.

**Design correction worth noting:** a naive weighted average (using every rating as a positive
weight) has a real flaw -- it can't distinguish "items I loved" from "items I merely tolerated."
An engineer who rates many mediocre items a "2" and only one great-fit item a "4" ends up with a
profile dominated by the *volume* of mediocre items, not the one strong signal. The fix is
**mean-centering**: only ratings *above* the scale midpoint (3) are allowed to pull the profile,
so lukewarm ratings don't drown out genuine preference. This is a standard technique in real
content-based systems, and it measurably improves profile accuracy here (verified below).

In [ ]:
def build_user_profile(user_idx):
    user_ratings = ratings[user_idx]
    # only ratings ABOVE the scale midpoint (3) contribute -- a "2" shouldn't
    # pull the profile toward that item's features just because it's nonzero
    weight = np.clip(user_ratings - 3.0, 0, None)
    if weight.sum() == 0:  # edge case: engineer rated nothing above midpoint
        weight = np.clip(user_ratings, 0, None)
    return (item_features * weight[:, None]).sum(axis=0) / weight.sum()

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

def content_recommend(user_idx, top_n=5):
    profile = build_user_profile(user_idx)
    already_rated = ratings[user_idx] > 0
    sims = np.array([cosine_sim(profile, item_features[i]) for i in range(n_items)])
    sims[already_rated] = -1
    top_idx = np.argsort(sims)[::-1][:top_n]
    return [(names[i], round(sims[i], 3)) for i in top_idx], profile

cb_recs, cb_profile = content_recommend(target_idx)

feature_labels = ["resolution", "frame_rate", "is_3D", "built_in_light", "ai_capable", "price_tier"]
print(f"Target engineer: {target}")
print("\nImplied profile (weighted-avg feature values, 0-1 scale):")
for label, val in zip(feature_labels, cb_profile):
    print(f"  {label}: {val:.2f}")
print("Compare to engineer's TRUE hidden target profile:")
for label, val in zip(feature_labels, profiles[target][0]):
    print(f"  {label}: {val:.2f}")
print(f"\nTop-5 content-based recommendations for {target}:")
for name, sim in cb_recs:
    print(f"  {name}: similarity {sim}")


## 5. Comparison: CF vs. Content-Based

Same target engineer, both methods, side by side.

In [ ]:
cf_names = [r[0] for r in cf_recs]
cb_names = [r[0] for r in cb_recs]
overlap = set(cf_names) & set(cb_names)

print(f"CF top-5:      {cf_names}")
print(f"Content top-5: {cb_names}")
print(f"Overlap ({len(overlap)} item(s)): {overlap}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_cf = ["#2ca02c" if n in overlap else "#1f77b4" for n in cf_names]
colors_cb = ["#2ca02c" if n in overlap else "#ff7f0e" for n in cb_names]

axes[0].barh(cf_names[::-1], [r[1] for r in cf_recs][::-1], color=colors_cf[::-1])
axes[0].set_title(f"Collaborative Filtering\nTop-5 for {target}")
axes[0].set_xlabel("Predicted rating")

axes[1].barh(cb_names[::-1], [r[1] for r in cb_recs][::-1], color=colors_cb[::-1])
axes[1].set_title(f"Content-Based Filtering\nTop-5 for {target}")
axes[1].set_xlabel("Cosine similarity to profile")

fig.suptitle("CF vs. Content-Based recommendations (green = appears in both)")
plt.tight_layout()
plt.show()


### Commentary: where they agree, where they differ, and what it reveals

- **Content-based filtering** stays tightly anchored to the individual engineer's own corrected
  profile: E7 (the high-resolution specialist) is recommended almost exclusively from the
  **2D Vision Sensor** family -- the one genuinely high-resolution product line in the catalog
  (7-10MP) -- while E3 (budget-focused) is recommended low-price **2D Sensor Entry / IVC-2D**
  items. This is content-based filtering doing exactly what it should: matching an individual's
  demonstrated preference, even a narrow or unusual one.
- **Collaborative filtering** tells a different story for the same two engineers: E7's CF
  recommendations are still IVC-2D/entry-level items -- essentially the *opposite* of what a
  high-resolution specialist wants. This exposes CF's real weakness in a sparse, niche dataset:
  E7 only overlaps with other engineers on a handful of items, so their "top-3 similar engineers"
  aren't a good match, and CF ends up recommending whatever those weakly-correlated neighbors
  happened to like. This is the classic **cold-start / sparsity problem** for collaborative
  filtering, and it's especially relevant for a B2B domain like this one where any single engineer
  has only worked a small number of past projects.
- **Overlap varies by engineer** (0-2 items out of 5) rather than following one fixed pattern --
  which itself is a useful finding: agreement between the two methods is highest when an engineer's
  taste happens to align with the broader group (E1, E8), and lowest when an engineer's taste is
  unusual relative to their peers (E3, E7). This is a strong, concrete argument for a **hybrid**
  approach: content-based filtering protects against CF's sparsity weakness for niche specialists,
  while CF adds useful signal (and diversity) when peer data is actually informative -- and a
  knowledge-based hard filter on top of both would guarantee that whatever gets recommended at
  least meets the physical requirements of the task, regardless of which method produced it.

## 6. Does the pattern hold across other engineers?

One target engineer isn't enough to claim a general pattern. Here we repeat the same CF vs.
content-based comparison for three engineers with very different task profiles: a budget-focused
retrofitter, a high-resolution defect specialist, and a fast 3D conveyor specialist -- to check
whether "CF = more diverse, content-based = more narrowly specialized" actually generalizes,
or was just a coincidence for E1.

In [ ]:
test_engineers = ["E3_BudgetLineRetrofit", "E7_HighResFineDefect", "E8_FastConveyor3D"]

overlap_counts = []
for eng in test_engineers:
    idx = engineers.index(eng)
    cf_r, _ = cf_recommend(idx)
    cb_r, _ = content_recommend(idx)
    cf_n = [r[0] for r in cf_r]
    cb_n = [r[0] for r in cb_r]
    ov = set(cf_n) & set(cb_n)
    overlap_counts.append(len(ov))

    print(f"\n{eng}")
    print(f"  CF top-5: {cf_n}")
    print(f"  CB top-5: {cb_n}")
    print(f"  Overlap: {len(ov)} -> {ov}")

print(f"\n\nOverlap across all 4 tested engineers (E1, E3, E7, E8): "
      f"{[len(overlap)] + overlap_counts} out of 5 possible each")


**Finding:** overlap between CF and content-based varies from 0 to 2 items depending on the
engineer -- it is NOT a fixed pattern, and that variation is itself informative (see commentary
below). This confirms the E1 result wasn't a coincidence, but it also revealed something more
interesting: CF specifically struggles for engineers whose taste is unusual relative to the group
(E3, E7), which is a concrete, data-backed argument for why a hybrid approach would outperform
either method alone in this domain.